# Notebook 05 - Argus Vision: Full Evaluation Report (23-dim consensus)

Complete evaluation of the Argus Vision dermoscopy system on ISIC-2019. It compares five configurations (Agent A, Agent B, Standard Ensemble, Deep Ensemble, Argus full), then runs the cost-sensitive malignant-vs-benign metric, a selective-prediction (abstention) sweep, bootstrap confidence intervals, and an error analysis.

**The Groq LLM debate text + 788-dim sentence-embedding pipeline has been removed.** Argus now predicts with the calibrated consensus MLP over the **23-dim numerical feature vector** from `extract_consensus_features` (identical to NB04 and `backend/ml/debate/features.py`). Features are standardized with the saved `consensus_scaler.pkl`.

## Kaggle setup
1. Add Data -> `andrewmvd/isic-2019`.
2. Settings -> Accelerator -> GPU; Internet -> ON (pip + pretrained timm weights).
3. Attach the previous notebooks' outputs so `find_file` can locate them:
   - NB01 `agent_a_best.pth`, NB02 `agent_b_best.pth`
   - NB04 `consensus_best.pth`, `consensus_scaler.pkl` (+ `.json`), `consensus_temperature.txt`
   - NB03 `hard_subset.csv`
4. Run All. (No Groq / sentence-transformers / API key is needed.)

In [ ]:
# ============================================================================
# Cell 2 — Install extras, imports, shared constants and discovery helpers
# ============================================================================
import sys, subprocess

# Kaggle pre-installs torch / torchvision / numpy / pandas / sklearn / matplotlib /
# seaborn / opencv-python / Pillow / tqdm. We add the extras THIS notebook needs.
# NOTE: Internet MUST be ON for these installs and for pretrained weight downloads.
print("Installing extras (Internet must be ON)...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", "timm"],
    check=False,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "grad-cam", "netcal", "scipy", "joblib", "scikit-learn"],
    check=False,
)
print("Extras installed (timm -U, grad-cam, netcal, scipy, joblib, scikit-learn). No Groq / sentence-transformers used.")

import os
import re
import json
import glob
import warnings
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# ---------------------------------------------------------------- Shared contract
ISIC_CLASSES: List[str] = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]
CLASS_NAMES = ISIC_CLASSES  # alias used throughout
FULL_NAMES: Dict[str, str] = {
    "MEL": "Melanoma",
    "NV": "Melanocytic Nevus",
    "BCC": "Basal Cell Carcinoma",
    "AK": "Actinic Keratosis",
    "BKL": "Benign Keratosis",
    "DF": "Dermatofibroma",
    "VASC": "Vascular Lesion",
    "SCC": "Squamous Cell Carcinoma",
}
RISK_LEVELS: Dict[str, str] = {
    "MEL": "high", "NV": "low", "BCC": "high", "AK": "medium",
    "BKL": "low", "DF": "low", "VASC": "medium", "SCC": "high",
}
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMAGE_SIZE = 224
NUM_CLASSES = 8
# Consensus feature contract (must match NB04 + backend/ml/debate/features.py).
FEATURE_DIM = 23

# Debate-trigger thresholds (identical to the backend settings).
TAU_JS = 0.25
TAU_ENTROPY = 0.8

# Backbone identifiers (MUST match the backend agents / NB01 + NB02).
AGENT_A_MODEL_NAME = "efficientnet_b4"
AGENT_B_MODEL_NAME = "vit_base_patch16_224.augreg_in21k_ft_in1k"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

BATCH_SIZE = 32          # lower to 16 if OOM
NUM_WORKERS = 2

WORK_DIR = Path("/kaggle/working")
FIG_DIR = WORK_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Feature dimension: {FEATURE_DIM}")


# ---------------------------------------------------------------- discover_isic()
def discover_isic(root: str = "/kaggle/input") -> Tuple[str, str]:
    """Locate the ISIC-2019 ground-truth CSV and the image directory under /kaggle/input.

    The CSV is the .csv whose header contains ALL 8 ISIC class names. The image_dir
    is the directory containing the most ISIC_*.jpg/.jpeg files (case-insensitive).
    Robust to nested mirror folders (e.g. double-nested ISIC_2019_Training_Input/).
    """
    needed = set(c.upper() for c in ISIC_CLASSES)
    csv_path: Optional[str] = None
    for dirpath, _dirs, files in os.walk(root):
        for fname in files:
            if not fname.lower().endswith(".csv"):
                continue
            fpath = os.path.join(dirpath, fname)
            try:
                header = pd.read_csv(fpath, nrows=0)
            except Exception:
                continue
            cols = set(str(c).upper() for c in header.columns)
            if needed.issubset(cols):
                csv_path = fpath
                break
        if csv_path is not None:
            break

    # Find the directory holding the most ISIC_*.jpg/.jpeg images.
    best_dir: Optional[str] = None
    best_count = -1
    for dirpath, _dirs, files in os.walk(root):
        cnt = 0
        for fname in files:
            low = fname.lower()
            if low.startswith("isic_") and (low.endswith(".jpg") or low.endswith(".jpeg")):
                cnt += 1
        if cnt > best_count:
            best_count = cnt
            best_dir = dirpath
    image_dir = best_dir

    print(f"discover_isic -> csv_path = {csv_path}")
    print(f"discover_isic -> image_dir = {image_dir}  ({best_count} ISIC_*.jpg/.jpeg files)")
    assert csv_path is not None, "Ground-truth CSV with all 8 ISIC class columns not found under /kaggle/input."
    assert image_dir is not None and best_count > 0, "No ISIC_*.jpg image directory found under /kaggle/input."
    assert os.path.exists(csv_path) and os.path.isdir(image_dir)
    return csv_path, image_dir


# ---------------------------------------------------------------- find_file()
def find_file(filename_substring: str,
              search_roots: Tuple[str, ...] = ("/kaggle/input", "/kaggle/working")) -> Optional[str]:
    """Return the first path whose basename contains `filename_substring`, else None."""
    for root in search_roots:
        if not os.path.isdir(root):
            continue
        for dirpath, _dirs, files in os.walk(root):
            for fname in files:
                if filename_substring in fname:
                    found = os.path.join(dirpath, fname)
                    print(f"find_file('{filename_substring}') -> {found}")
                    return found
    print(f"find_file('{filename_substring}') -> NOT FOUND under {search_roots}.")
    return None


CSV_PATH, IMAGE_DIR = discover_isic()
print("Setup complete.")

## Step 1 - Load both agents, the consensus MLP (+ temperature) and the feature scaler

Agent A is EfficientNet-B4, Agent B is ViT-B/16. The consensus head is a 23-dim MLP with a learnable temperature. We also load the `StandardScaler` (`consensus_scaler.pkl`, with a `.json` sidecar fallback) fit on the consensus training split. The sentence encoder is gone.

In [ ]:
import timm
import joblib
from sklearn.preprocessing import StandardScaler


class ConsensusMLP(nn.Module):
    """23-dim consensus classifier. Parameter names (mlp.*, temperature) match
    NB04 and backend/ml/consensus/classifier.py so the checkpoint loads."""

    def __init__(self, input_dim: int = FEATURE_DIM, num_classes: int = NUM_CLASSES,
                 dropout: float = 0.3) -> None:
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes),
        )
        self.temperature = nn.Parameter(torch.ones(1))

    def logits(self, features: torch.Tensor) -> torch.Tensor:
        return self.mlp(features)

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return self.logits(features) / self.temperature.clamp_min(1e-2)


class _NumpyScaler:
    """Minimal StandardScaler stand-in (.transform) for the json-sidecar / identity path."""

    def __init__(self, mean, scale):
        self.mean_ = np.asarray(mean, dtype=np.float64)
        self.scale_ = np.where(np.asarray(scale, dtype=np.float64) == 0.0, 1.0,
                               np.asarray(scale, dtype=np.float64))

    def transform(self, X):
        return (np.asarray(X, dtype=np.float64) - self.mean_) / self.scale_


def build_agent(model_name: str, ckpt_substring: str):
    """Create a timm backbone and load its fine-tuned checkpoint if found."""
    ckpt = find_file(ckpt_substring)
    has_ckpt = ckpt is not None
    model = timm.create_model(model_name, pretrained=not has_ckpt, num_classes=NUM_CLASSES)
    if has_ckpt:
        state = torch.load(ckpt, map_location=DEVICE)
        if isinstance(state, dict) and "state_dict" in state:
            state = state["state_dict"]
        missing, unexpected = model.load_state_dict(state, strict=False)
        print(f"Loaded {os.path.basename(ckpt)} (missing={len(missing)}, unexpected={len(unexpected)}).")
    else:
        print(f"WARNING: no checkpoint matching '{ckpt_substring}'; using ImageNet fallback.")
    return model.eval().to(DEVICE), has_ckpt


agent_a, A_HAS_CKPT = build_agent(AGENT_A_MODEL_NAME, "agent_a_best")
agent_b, B_HAS_CKPT = build_agent(AGENT_B_MODEL_NAME, "agent_b_best")

# ---- Consensus MLP + temperature -------------------------------------------
consensus = ConsensusMLP().to(DEVICE)
CONSENSUS_HAS_CKPT = False
consensus_ckpt = find_file("consensus_best")
if consensus_ckpt is not None:
    cstate = torch.load(consensus_ckpt, map_location=DEVICE)
    if isinstance(cstate, dict) and "state_dict" in cstate:
        cstate = cstate["state_dict"]
    res = consensus.load_state_dict(cstate, strict=False)
    CONSENSUS_HAS_CKPT = True
    print(f"Loaded consensus checkpoint: {os.path.basename(consensus_ckpt)} "
          f"(missing={len(res.missing_keys)}, unexpected={len(res.unexpected_keys)}).")
else:
    print("WARNING: consensus_best.pth not found; consensus head is randomly initialised.")
consensus.eval()

# Temperature sidecar overrides the in-state value if present.
CONSENSUS_TEMPERATURE = float(consensus.temperature.detach().cpu().item())
temp_json = find_file("consensus_temperature")
if temp_json is not None:
    try:
        with open(temp_json, "r", encoding="utf-8") as fh:
            content = fh.read().strip()
        try:
            CONSENSUS_TEMPERATURE = float(content)
        except ValueError:
            tmeta = json.loads(content)
            CONSENSUS_TEMPERATURE = float(tmeta.get("temperature", CONSENSUS_TEMPERATURE)
                                          if isinstance(tmeta, dict) else tmeta)
        with torch.no_grad():
            consensus.temperature.fill_(CONSENSUS_TEMPERATURE)
        print(f"Applied temperature from sidecar: T={CONSENSUS_TEMPERATURE:.4f}")
    except Exception as exc:
        print(f"Could not read temperature sidecar ({exc}); using T={CONSENSUS_TEMPERATURE:.4f}.")
else:
    print(f"No temperature sidecar; using in-checkpoint T={CONSENSUS_TEMPERATURE:.4f}.")

# ---- Feature scaler (REQUIRED: the MLP was trained on standardized features) ----
scaler = None
scaler_pkl = find_file("consensus_scaler.pkl")
if scaler_pkl is not None:
    try:
        scaler = joblib.load(scaler_pkl)
        _ = scaler.transform(np.zeros((1, FEATURE_DIM)))  # smoke test
        print(f"Loaded StandardScaler from {os.path.basename(scaler_pkl)}.")
    except Exception as exc:
        print(f"Could not load joblib scaler ({exc}); trying JSON sidecar.")
        scaler = None
if scaler is None:
    scaler_json = find_file("consensus_scaler.json")
    if scaler_json is not None:
        with open(scaler_json, "r", encoding="utf-8") as fh:
            payload = json.load(fh)
        scaler = _NumpyScaler(payload["mean"], payload["scale"])
        print(f"Loaded scaler from {os.path.basename(scaler_json)} (numpy fallback).")
if scaler is None:
    scaler = _NumpyScaler(np.zeros(FEATURE_DIM), np.ones(FEATURE_DIM))
    print("WARNING: no consensus_scaler found -> using IDENTITY scaler. Consensus predictions "
          "will be miscalibrated because the MLP was trained on standardized features. "
          "Attach NB04's consensus_scaler.pkl / .json.")

print("All models + scaler ready on", DEVICE)


## Step 2 — Build the test split (same `random_state=42`)

We read the one-hot ISIC-2019 ground-truth CSV (`label = argmax` over the 8 class columns),
resolve image paths against the discovered `image_dir`, and reproduce a **stratified
85/15 split** with `train_test_split(test_size=0.15, stratify=labels, random_state=42)`.
Evaluation runs on the held-out 15% test split. The deterministic eval transform is
`Resize(256) -> CenterCrop(224) -> ToTensor -> Normalize(ImageNet)`.

In [ ]:
# ============================================================================
# Cell 4 — ISIC dataframe + stratified test split + eval transform / loaders
# ============================================================================
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# Read the one-hot ground-truth CSV; label = argmax over the 8 class columns.
gt = pd.read_csv(CSV_PATH)
image_col = "image" if "image" in gt.columns else gt.columns[0]
col_upper = {str(c).upper(): c for c in gt.columns}
onehot_cols = [col_upper[c.upper()] for c in ISIC_CLASSES]
labels_all = gt[onehot_cols].to_numpy().argmax(axis=1).astype(int)


def resolve_image_path(stem: str) -> str:
    """Resolve an ISIC image stem to a path inside IMAGE_DIR (tries .jpg/.jpeg)."""
    for ext in (".jpg", ".jpeg", ".JPG", ".JPEG", ".png", ".PNG"):
        p = os.path.join(IMAGE_DIR, str(stem) + ext)
        if os.path.isfile(p):
            return p
    return os.path.join(IMAGE_DIR, str(stem) + ".jpg")


records = []
for i in range(len(gt)):
    stem = str(gt.iloc[i][image_col])
    records.append({"image_path": resolve_image_path(stem),
                    "image_id": stem,
                    "label": int(labels_all[i])})
full_df = pd.DataFrame.from_records(records).reset_index(drop=True)

# Keep only rows whose image actually exists on disk.
exists_mask = full_df["image_path"].map(os.path.isfile)
n_missing = int((~exists_mask).sum())
if n_missing:
    print(f"Note: dropping {n_missing} rows whose image file was not found on disk.")
full_df = full_df[exists_mask].reset_index(drop=True)
print(f"Usable labelled images: {len(full_df):,}")

train_df, test_df = train_test_split(
    full_df, test_size=0.15, stratify=full_df["label"], random_state=SEED
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
print(f"Train: {len(train_df):,} | Test (held-out 15%): {len(test_df):,}")

eval_transform = T.Compose([
    T.Resize(256),
    T.CenterCrop(IMAGE_SIZE),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


class ISICEvalDataset(Dataset):
    """Returns (tensor, label_int, image_path) for the eval split."""

    def __init__(self, frame: pd.DataFrame, transform):
        self.paths = frame["image_path"].tolist()
        self.labels = frame["label"].tolist()
        self.transform = transform

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int):
        path = self.paths[idx]
        with Image.open(path) as h:
            img = h.convert("RGB")
        return self.transform(img), int(self.labels[idx]), path


def load_tensor(image_path: str) -> torch.Tensor:
    """Load one image as a (1,3,224,224) eval tensor (used for single-image debate)."""
    with Image.open(image_path) as h:
        img = h.convert("RGB")
    return eval_transform(img).unsqueeze(0)


# Optional cap to keep the full-split evaluation tractable on a Kaggle session.
# Set MAX_EVAL_IMAGES = None to evaluate the entire test split.
MAX_EVAL_IMAGES = 2000
if MAX_EVAL_IMAGES is not None and len(test_df) > MAX_EVAL_IMAGES:
    eval_df = test_df.groupby("label", group_keys=False).apply(
        lambda g: g.sample(min(len(g), max(1, MAX_EVAL_IMAGES // NUM_CLASSES)),
                            random_state=SEED)
    ).reset_index(drop=True)
    print(f"Evaluating on a stratified subset of {len(eval_df):,} test images "
          f"(set MAX_EVAL_IMAGES=None for the full split).")
else:
    eval_df = test_df.copy()
    print(f"Evaluating on the full test split: {len(eval_df):,} images.")

eval_ds = ISICEvalDataset(eval_df, eval_transform)
eval_loader = DataLoader(eval_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))
print("Eval loader ready.")

## Step 3 - Inline the numerical consensus pipeline (attention, trigger, 23-dim features)

These mirror the backend / NB04 modules exactly: Grad-CAM++ (Agent A), attention rollout (Agent B), the trigger (JS divergence / entropy), and the canonical `extract_consensus_features` 23-dim builder. No LLM debate, no sentence embeddings.

In [ ]:
import cv2
from scipy.spatial.distance import jensenshannon
from scipy.stats import entropy as scipy_entropy
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

_EPS = 1e-12
GRID_SIZE = 14
NUM_PATCH_TOKENS = GRID_SIZE * GRID_SIZE


def shannon_entropy(probs):
    p = np.clip(probs, 0.0, None)
    return -float(np.sum(p * np.log2(p + _EPS)))


def js_divergence(pa, pb):
    d = jensenshannon(pa, pb, base=2)
    div = float(d) ** 2
    return div if np.isfinite(div) else 0.0


def trigger_fires(div, ent_a, ent_b):
    return (div > TAU_JS) or (max(ent_a, ent_b) > TAU_ENTROPY)


@torch.no_grad()
def predict_probs(model, tensor):
    logits = model(tensor.to(DEVICE))
    return F.softmax(logits, dim=1)[0].detach().cpu().numpy().astype(np.float64)


def gradcam_plusplus(model, tensor, target):
    """224x224 Grad-CAM++ saliency map for Agent A (EfficientNet-B4), in [0,1]."""
    blocks = getattr(model, "blocks", None)
    if blocks is not None and len(blocks) > 0:
        last = blocks[-1]
        target_layer = getattr(last, "bn3", last)
    else:
        target_layer = getattr(model, "conv_head", None)
        if target_layer is None:
            for m in model.modules():
                if isinstance(m, (nn.BatchNorm2d, nn.Conv2d)):
                    target_layer = m
    model.zero_grad()
    cam = GradCAMPlusPlus(model=model, target_layers=[target_layer])
    grayscale = cam(input_tensor=tensor.to(DEVICE), targets=[ClassifierOutputTarget(int(target))])
    model.zero_grad()
    hm = np.asarray(grayscale[0], dtype=np.float32)
    if hm.shape != (IMAGE_SIZE, IMAGE_SIZE):
        hm = cv2.resize(hm, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_LINEAR)
    return np.clip(hm, 0.0, 1.0).astype(np.float32)


def attention_rollout(model, tensor):
    """224x224 attention-rollout saliency map for Agent B (ViT-B/16), in [0,1]."""
    handles, captured, original_fused = [], {}, {}
    blocks = getattr(model, "blocks", None)

    def make_hook(layer_idx):
        def hook(module, inputs, output):
            x = inputs[0]
            batch, num_tokens, dim = x.shape
            heads = int(module.num_heads)
            head_dim = dim // heads
            scale = getattr(module, "scale", head_dim ** -0.5)
            qkv = module.qkv(x).reshape(batch, num_tokens, 3, heads, head_dim)
            qkv = qkv.permute(2, 0, 3, 1, 4)
            q, k = qkv[0], qkv[1]
            attn = (q @ k.transpose(-2, -1)) * float(scale)
            attn = attn.softmax(dim=-1)
            captured[layer_idx] = attn.mean(dim=1).detach().to(torch.float32).cpu()
        return hook

    try:
        for idx, block in enumerate(blocks):
            am = getattr(block, "attn", None)
            if am is None or not hasattr(am, "qkv"):
                continue
            original_fused[idx] = bool(getattr(am, "fused_attn", False))
            if hasattr(am, "fused_attn"):
                am.fused_attn = False
            handles.append(am.register_forward_hook(make_hook(idx)))
        with torch.no_grad():
            model(tensor.to(DEVICE))
        layer_indices = sorted(captured.keys())
        num_tokens = captured[layer_indices[0]].shape[-1]
        identity = torch.eye(num_tokens, dtype=torch.float32)
        rollout = torch.eye(num_tokens, dtype=torch.float32)
        for idx in layer_indices:
            attn = captured[idx][0]
            aug = 0.5 * attn + 0.5 * identity
            aug = aug / aug.sum(dim=-1, keepdim=True).clamp_min(1e-12)
            rollout = aug @ rollout
        cls_attention = rollout[0, 1:1 + NUM_PATCH_TOKENS]
        grid = cls_attention.reshape(GRID_SIZE, GRID_SIZE).numpy().astype(np.float32)
        hm = cv2.resize(grid, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_CUBIC).astype(np.float32)
        span = float(hm.max() - hm.min())
        if span < 1e-12:
            return np.zeros((IMAGE_SIZE, IMAGE_SIZE), dtype=np.float32)
        return ((hm - hm.min()) / span).astype(np.float32)
    finally:
        for h in handles:
            h.remove()
        for idx, was_fused in original_fused.items():
            am = getattr(blocks[idx], "attn", None)
            if am is not None and hasattr(am, "fused_attn"):
                am.fused_attn = was_fused


def _normalize_map(arr):
    arr = np.asarray(arr, dtype=np.float32)
    span = float(arr.max() - arr.min())
    if span < 1e-12:
        return np.zeros_like(arr, dtype=np.float32)
    return ((arr - arr.min()) / span).astype(np.float32)


def disagreement_map(heatmap_a, heatmap_b):
    """M_delta = |normalised(A) - normalised(B)| in [0,1] (for case-study viz)."""
    return np.abs(_normalize_map(heatmap_a) - _normalize_map(heatmap_b)).astype(np.float32)


# --- Canonical 23-dim consensus feature extractor (IDENTICAL to NB04 + backend) ---
FEATURE_NAMES = (
    [f"pA_{c}" for c in CLASS_NAMES]
    + [f"pB_{c}" for c in CLASS_NAMES]
    + ["js_div", "entropy_a", "entropy_b", "max_prob_delta",
       "attn_iou", "attn_entropy_a", "attn_entropy_b"]
)


def extract_consensus_features(prob_a, prob_b, attn_map_a=None, attn_map_b=None):
    prob_a = np.asarray(prob_a, dtype=np.float64).flatten()
    prob_b = np.asarray(prob_b, dtype=np.float64).flatten()

    prob_a = np.clip(prob_a, 1e-9, 1.0)
    prob_a /= prob_a.sum()
    prob_b = np.clip(prob_b, 1e-9, 1.0)
    prob_b /= prob_b.sum()

    js_div = float(jensenshannon(prob_a, prob_b) ** 2)
    entropy_a = float(scipy_entropy(prob_a, base=2))
    entropy_b = float(scipy_entropy(prob_b, base=2))
    max_prob_delta = float(np.max(np.abs(prob_a - prob_b)))

    if attn_map_a is not None and attn_map_b is not None:
        a = np.asarray(attn_map_a, dtype=np.float32)
        b = np.asarray(attn_map_b, dtype=np.float32)
        a = (a - a.min()) / (a.max() - a.min() + 1e-9)
        b = (b - b.min()) / (b.max() - b.min() + 1e-9)
        mask_a = (a >= 0.5).astype(np.float32)
        mask_b = (b >= 0.5).astype(np.float32)
        intersection = (mask_a * mask_b).sum()
        union = np.clip(mask_a + mask_b, 0, 1).sum()
        attn_iou = float(intersection / (union + 1e-9))
        a_flat = a.flatten() + 1e-9
        a_flat /= a_flat.sum()
        b_flat = b.flatten() + 1e-9
        b_flat /= b_flat.sum()
        attn_entropy_a = float(scipy_entropy(a_flat, base=2))
        attn_entropy_b = float(scipy_entropy(b_flat, base=2))
    else:
        attn_iou = 0.0
        attn_entropy_a = 0.0
        attn_entropy_b = 0.0

    feat = np.concatenate([
        prob_a,
        prob_b,
        [js_div, entropy_a, entropy_b, max_prob_delta,
         attn_iou, attn_entropy_a, attn_entropy_b],
    ]).astype(np.float32)

    assert feat.shape == (FEATURE_DIM,), \
        f"Feature dim mismatch: got {feat.shape}, expected ({FEATURE_DIM},)"
    assert not np.any(np.isnan(feat)), "NaN in feature vector"
    assert not np.any(np.isinf(feat)), "Inf in feature vector"
    return feat


print("Pipeline / attention / 23-dim feature helpers ready.")


## Step 4 - Evaluate FIVE configurations on the test split

Per image we compute pA and pB once, then derive: (1) Agent A, (2) Agent B, (3) Standard ensemble = mean(pA, pB), (4) Deep ensemble (if a second seed's checkpoints are attached, else mirrors the standard ensemble and is flagged), and (5) **Argus (full)** = the calibrated consensus MLP on the standardized 23-dim feature vector. Attention features are computed when the trigger fires, else 0.0. The consensus head runs on every image so that evaluation matches the backend serving path exactly.

In [ ]:
from tqdm.auto import tqdm

# Optional second checkpoint set for a DEEP ENSEMBLE.
deep_a_path = find_file("agent_a_seed")
deep_b_path = find_file("agent_b_seed")
DEEP_ENSEMBLE_AVAILABLE = (deep_a_path is not None) and (deep_b_path is not None)
deep_agent_a = deep_agent_b = None
if DEEP_ENSEMBLE_AVAILABLE:
    try:
        da = timm.create_model(AGENT_A_MODEL_NAME, pretrained=False, num_classes=NUM_CLASSES)
        sa = torch.load(deep_a_path, map_location=DEVICE)
        da.load_state_dict(sa.get("state_dict", sa) if isinstance(sa, dict) else sa, strict=False)
        deep_agent_a = da.eval().to(DEVICE)
        db = timm.create_model(AGENT_B_MODEL_NAME, pretrained=False, num_classes=NUM_CLASSES)
        sb = torch.load(deep_b_path, map_location=DEVICE)
        db.load_state_dict(sb.get("state_dict", sb) if isinstance(sb, dict) else sb, strict=False)
        deep_agent_b = db.eval().to(DEVICE)
        print("Deep-ensemble second checkpoints loaded.")
    except Exception as exc:
        print(f"Deep-ensemble load failed ({exc}); will SKIP gracefully.")
        DEEP_ENSEMBLE_AVAILABLE = False
else:
    print("Deep-ensemble checkpoints (agent_*_seed*) not attached -> SKIPPING "
          "(column mirrors the standard ensemble and is flagged in notes).")

CONFIG_NAMES = ["Agent A", "Agent B", "Standard Ensemble", "Deep Ensemble", "Argus (full)"]
y_true, image_paths = [], []
proba = {name: [] for name in CONFIG_NAMES}
debate_records = []


@torch.no_grad()
def batch_probs(model, x):
    return F.softmax(model(x.to(DEVICE)), dim=1).detach().cpu().numpy().astype(np.float64)


def argus_predict(image_path, pa, pb):
    """Argus prediction = calibrated consensus MLP on the standardized 23-dim vector.

    Attention features are computed when the trigger fires, else 0.0 (matching the
    backend). The consensus head runs on EVERY image so eval == serve.
    """
    div = js_divergence(pa, pb)
    ent_a, ent_b = shannon_entropy(pa), shannon_entropy(pb)
    fired = trigger_fires(div, ent_a, ent_b)
    rec = {"image_path": image_path, "image_id": Path(image_path).stem,
           "js": div, "ent_a": ent_a, "ent_b": ent_b, "fired": bool(fired),
           "pa": pa, "pb": pb}
    if fired:
        tensor = load_tensor(image_path)
        hm_a = gradcam_plusplus(agent_a, tensor, int(pa.argmax()))
        hm_b = attention_rollout(agent_b, tensor)
    else:
        hm_a = hm_b = None
    feat = extract_consensus_features(pa, pb, hm_a, hm_b)
    feat_scaled = np.asarray(scaler.transform(feat.reshape(1, -1))[0], dtype=np.float32)
    consensus.eval()
    with torch.no_grad():
        ft = torch.from_numpy(feat_scaled).float().unsqueeze(0).to(DEVICE)
        cprob = F.softmax(consensus(ft), dim=1)[0].detach().cpu().numpy().astype(np.float64)
    # Report the JS value the MLP actually saw (feat[16]) so error analysis / case
    # studies match the feature vector; the trigger decision above keeps its own base-2 div.
    rec.update({"feat": feat, "js": float(feat[16]), "attn_iou": float(feat[20]), "consensus_prob": cprob})
    return cprob, rec


print("Running five-configuration evaluation (23-dim consensus)...")
for x, yb, paths in tqdm(eval_loader, total=len(eval_loader)):
    pA = batch_probs(agent_a, x)
    pB = batch_probs(agent_b, x)
    if DEEP_ENSEMBLE_AVAILABLE:
        pA2 = batch_probs(deep_agent_a, x)
        pB2 = batch_probs(deep_agent_b, x)

    for j in range(x.shape[0]):
        pa, pb = pA[j], pB[j]
        path = paths[j]
        y_true.append(int(yb[j]))
        image_paths.append(path)
        proba["Agent A"].append(pa)
        proba["Agent B"].append(pb)
        std_ens = 0.5 * (pa + pb)
        proba["Standard Ensemble"].append(std_ens)
        if DEEP_ENSEMBLE_AVAILABLE:
            proba["Deep Ensemble"].append(0.25 * (pa + pb + pA2[j] + pB2[j]))
        else:
            proba["Deep Ensemble"].append(std_ens)
        argus_p, rec = argus_predict(path, pa, pb)
        proba["Argus (full)"].append(argus_p)
        debate_records.append(rec)

y_true = np.asarray(y_true, dtype=np.int64)
for name in CONFIG_NAMES:
    proba[name] = np.asarray(proba[name], dtype=np.float64)

n_fired = int(sum(r["fired"] for r in debate_records))
print(f"Done. Evaluated {len(y_true):,} images; trigger fired on "
      f"{n_fired:,} ({n_fired / max(len(y_true), 1):.1%}).")


## Step 5 — Metrics table: Balanced Accuracy, Macro AUC, ECE per configuration

- **Balanced Accuracy** — mean per-class recall (robust to ISIC's heavy imbalance).
- **Macro AUC** — one-vs-rest ROC-AUC averaged over the 8 classes.
- **ECE** — Expected Calibration Error of the predicted top-class probability (via `netcal`,
  with a simple binned fallback if `netcal` is unavailable).

In [ ]:
# ============================================================================
# Cell 7 — Metric helpers + the per-configuration metrics table.
# ============================================================================
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score
from sklearn.metrics import confusion_matrix, classification_report

try:
    from netcal.metrics import ECE as _NetcalECE
    _HAVE_NETCAL = True
except Exception:
    _HAVE_NETCAL = False


def expected_calibration_error(probs: np.ndarray, labels: np.ndarray, n_bins: int = 15) -> float:
    """ECE of the predicted top-class confidence. Uses netcal if available."""
    if _HAVE_NETCAL:
        try:
            return float(_NetcalECE(n_bins).measure(probs.astype(np.float64), labels.astype(int)))
        except Exception:
            pass
    conf = probs.max(axis=1)
    pred = probs.argmax(axis=1)
    correct = (pred == labels).astype(np.float64)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    n = len(labels)
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        m = (conf > lo) & (conf <= hi) if i > 0 else (conf >= lo) & (conf <= hi)
        if m.sum() == 0:
            continue
        ece += (m.sum() / n) * abs(correct[m].mean() - conf[m].mean())
    return float(ece)


def macro_auc(probs: np.ndarray, labels: np.ndarray) -> float:
    present = np.unique(labels)
    if len(present) < 2:
        return float("nan")
    try:
        return float(roc_auc_score(labels, probs, multi_class="ovr",
                                   average="macro", labels=list(range(NUM_CLASSES))))
    except Exception:
        aucs = []
        for c in range(NUM_CLASSES):
            yc = (labels == c).astype(int)
            if yc.sum() == 0 or yc.sum() == len(yc):
                continue
            aucs.append(roc_auc_score(yc, probs[:, c]))
        return float(np.mean(aucs)) if aucs else float("nan")


def metrics_for(probs: np.ndarray, labels: np.ndarray) -> Dict[str, float]:
    preds = probs.argmax(axis=1)
    return {
        "Accuracy": float(accuracy_score(labels, preds)),
        "Balanced Accuracy": float(balanced_accuracy_score(labels, preds)),
        "Macro AUC": macro_auc(probs, labels),
        "ECE": expected_calibration_error(probs, labels),
    }


rows = []
for name in CONFIG_NAMES:
    m = metrics_for(proba[name], y_true)
    note = ""
    if name == "Deep Ensemble" and not DEEP_ENSEMBLE_AVAILABLE:
        note = "SKIPPED (no 2nd checkpoints; mirrors Standard Ensemble)"
    rows.append({"Configuration": name, **m, "Note": note})

metrics_table = pd.DataFrame(rows).set_index("Configuration")
metrics_table_display = metrics_table.copy()
for c in ["Accuracy", "Balanced Accuracy", "Macro AUC", "ECE"]:
    metrics_table_display[c] = metrics_table_display[c].map(lambda v: f"{v:.4f}")
metrics_table.to_csv(WORK_DIR / "metrics_full_split.csv")
print("Full-split metrics (saved to /kaggle/working/metrics_full_split.csv):")
metrics_table_display

print("\n=== Argus (full) Classification Report ===")
argus_preds = proba["Argus (full)"].argmax(axis=1)
present_labels = sorted(list(set(y_true) | set(argus_preds)))
print(classification_report(y_true, argus_preds, labels=present_labels, target_names=[ISIC_CLASSES[i] for i in present_labels], zero_division=0))

print("\n=== Argus (full) Confusion Matrix ===")
cm = confusion_matrix(y_true, argus_preds, labels=list(range(NUM_CLASSES)))
print(pd.DataFrame(cm, index=ISIC_CLASSES, columns=ISIC_CLASSES))


## Phase 4 - Malignant vs benign (cost-sensitive) + MEL->NV rate
Clinically-motivated grouping: malignant = {MEL, BCC, AK, SCC}, benign = {NV, BKL, DF, VASC}. Reported as a primary metric alongside (not replacing) 8-class balanced accuracy. The MEL-predicted-as-NV rate is the most clinically dangerous confusion.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

MALIGNANT = {"MEL", "BCC", "AK", "SCC"}
BENIGN = {"NV", "BKL", "DF", "VASC"}


def to_binary(labels):
    return np.array([1 if CLASS_NAMES[int(l)] in MALIGNANT else 0 for l in labels])


print("=== Malignant vs benign (malignant = {MEL, BCC, AK, SCC}) ===")
mvb_rows = []
for name in CONFIG_NAMES:
    pred = proba[name].argmax(axis=1)
    bt, bp = to_binary(y_true), to_binary(pred)
    rec = recall_score(bt, bp, zero_division=0)
    prec = precision_score(bt, bp, zero_division=0)
    f1 = f1_score(bt, bp, zero_division=0)
    mvb_rows.append({"Configuration": name, "Malignant recall": rec,
                     "Malignant precision": prec, "Malignant F1": f1})
    print(f"  {name:<20} recall={rec:.4f}  precision={prec:.4f}  F1={f1:.4f}")
mvb_table = pd.DataFrame(mvb_rows).set_index("Configuration")
mvb_table.to_csv(WORK_DIR / "malignant_vs_benign.csv")

mel_idx = CLASS_NAMES.index("MEL")
nv_idx = CLASS_NAMES.index("NV")
print("\n=== MEL predicted as NV (most dangerous confusion) ===")
for name in ["Argus (full)", "Standard Ensemble", "Agent A", "Agent B"]:
    pred = proba[name].argmax(axis=1)
    mask = (y_true == mel_idx)
    n_mel = int(mask.sum())
    n_as_nv = int((pred[mask] == nv_idx).sum())
    rate = n_as_nv / n_mel if n_mel else 0.0
    print(f"  {name:<20} {n_as_nv}/{n_mel} = {rate:.1%}")


## Phase 5 - Selective prediction / abstention
Using Argus's calibrated top-class confidence, sweep an abstention threshold and plot coverage (fraction still classified) vs selective balanced accuracy on the retained subset. Below the threshold the system would output 'uncertain - refer for specialist review'.

In [ ]:
import matplotlib.pyplot as plt

argus_pred = proba["Argus (full)"].argmax(axis=1)
argus_confidences = proba["Argus (full)"].max(axis=1)

thresholds = np.arange(0.30, 0.91, 0.05)
coverages, sel_accs = [], []
for tau in thresholds:
    mask = argus_confidences >= tau
    if mask.sum() == 0:
        coverages.append(0.0); sel_accs.append(float("nan")); continue
    cov = float(mask.mean())
    sba = balanced_accuracy_score(y_true[mask], argus_pred[mask])
    coverages.append(cov); sel_accs.append(sba)
    print(f"tau={tau:.2f}  coverage={cov:.3f}  selective_bal_acc={sba:.4f}  "
          f"abstained={int((1 - cov) * len(y_true))}")

fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.plot(thresholds, coverages, "b-o", label="Coverage")
ax1.set_xlabel("Confidence threshold (tau)")
ax1.set_ylabel("Coverage (fraction classified)", color="b")
ax2 = ax1.twinx()
ax2.plot(thresholds, sel_accs, "r-s", label="Selective bal. acc.")
ax2.set_ylabel("Selective balanced accuracy", color="r")
ax1.set_title("Argus: coverage vs selective balanced accuracy")
fig.tight_layout()
plt.savefig(FIG_DIR / "abstention_curve.png", dpi=150)
plt.show()
print("Saved: figures/abstention_curve.png")

# Recommend the max selective balanced accuracy among thresholds retaining >=50% coverage.
valid = [(t, c, s) for t, c, s in zip(thresholds, coverages, sel_accs)
         if c >= 0.5 and not np.isnan(s)]
if valid:
    t_rec, c_rec, s_rec = max(valid, key=lambda z: z[2])
    basis = "max selective balanced accuracy among thresholds keeping >=50% coverage"
else:
    finite = [(t, c, s) for t, c, s in zip(thresholds, coverages, sel_accs) if not np.isnan(s)]
    t_rec, c_rec, s_rec = max(finite, key=lambda z: z[2])
    basis = "max selective balanced accuracy (no threshold kept >=50% coverage)"
print(f"\nRecommended threshold: tau={t_rec:.2f}")
print(f"  Coverage:                {c_rec:.3f}")
print(f"  Selective balanced acc.: {s_rec:.4f}")
print(f"  Basis: {basis}.")


## Phase 6 - Bootstrap 95% confidence intervals
1000-resample bootstrap CIs for the balanced accuracy of Argus (full) and the Standard Ensemble, with an explicit overlap check (overlapping intervals => the difference may not be statistically meaningful).

In [ ]:
def bootstrap_balanced_accuracy(y_t, y_p, n_resamples=1000, random_state=42):
    rng = np.random.RandomState(random_state)
    y_t = np.asarray(y_t); y_p = np.asarray(y_p)
    scores = []
    for _ in range(n_resamples):
        idx = rng.randint(0, len(y_t), size=len(y_t))
        scores.append(balanced_accuracy_score(y_t[idx], y_p[idx]))
    scores = np.asarray(scores)
    return float(scores.mean()), float(np.percentile(scores, 2.5)), float(np.percentile(scores, 97.5))


argus_pred = proba["Argus (full)"].argmax(axis=1)
ens_pred = proba["Standard Ensemble"].argmax(axis=1)
am, alo, ahi = bootstrap_balanced_accuracy(y_true, argus_pred)
em, elo, ehi = bootstrap_balanced_accuracy(y_true, ens_pred)

print("=== Bootstrap 95% CI for balanced accuracy (1000 resamples) ===")
print(f"Argus (full):      {am:.4f}  [{alo:.4f}, {ahi:.4f}]")
print(f"Standard Ensemble: {em:.4f}  [{elo:.4f}, {ehi:.4f}]")
overlap = alo < ehi and elo < ahi
print(f"\nIntervals overlap: {overlap}")
if overlap:
    print("  -> the Argus-vs-Ensemble difference may NOT be statistically significant.")
else:
    print("  -> the intervals do not overlap: the difference is statistically meaningful.")


## Phase 7 - Error analysis
Representative misclassified cases on the dangerous confusions (MEL->NV, AK->BCC, AK->BKL, SCC->BCC), plus 'debate-fail' cases where the Standard Ensemble was correct but Argus was wrong. For each, the 23-dim feature drivers (agent top classes, JS divergence, entropies, attention IoU) explain why the decision went wrong.

In [ ]:
import random

argus_pred = proba["Argus (full)"].argmax(axis=1)
ens_pred = proba["Standard Ensemble"].argmax(axis=1)
argus_confidences = proba["Argus (full)"].max(axis=1)

stem_to_idx = {Path(p).stem: i for i, p in enumerate(image_paths)}
rec_by_idx = {stem_to_idx[r["image_id"]]: r for r in debate_records}

TARGET_CONFUSIONS = [("MEL", "NV"), ("AK", "BCC"), ("AK", "BKL"), ("SCC", "BCC")]
print("=== Error analysis: representative misclassified cases ===\n")
random.seed(42)
for true_cls, pred_cls in TARGET_CONFUSIONS:
    ti, pi = CLASS_NAMES.index(true_cls), CLASS_NAMES.index(pred_cls)
    idxs = [i for i in range(len(y_true)) if y_true[i] == ti and argus_pred[i] == pi]
    if not idxs:
        print(f"{true_cls}->{pred_cls}: no cases found\n")
        continue
    sample = random.sample(idxs, min(2, len(idxs)))
    print(f"{true_cls}->{pred_cls}: {len(idxs)} total cases, showing {len(sample)}")
    for i in sample:
        r = rec_by_idx.get(i)
        pa, pb = r["pa"], r["pb"]
        print(f"  idx {i} ({r['image_id']}):")
        print(f"    Agent A top: {CLASS_NAMES[int(np.argmax(pa))]} ({pa.max():.3f})  |  "
              f"Agent B top: {CLASS_NAMES[int(np.argmax(pb))]} ({pb.max():.3f})")
        print(f"    JS={r['js']:.4f}  entropy_A={r['ent_a']:.2f}  entropy_B={r['ent_b']:.2f}  "
              f"attn_iou={r.get('attn_iou', 0.0):.3f}  fired={r['fired']}")
        print(f"    Argus confidence={argus_confidences[i]:.3f} (predicted {pred_cls}); true {true_cls}")
    print()

# Debate-fail cases: ensemble correct, Argus wrong.
fails = [i for i in range(len(y_true)) if ens_pred[i] == y_true[i] and argus_pred[i] != y_true[i]]
print(f"=== Debate-fail cases (Standard Ensemble correct, Argus wrong): "
      f"{len(fails)} of {len(y_true)} ({len(fails) / max(len(y_true), 1):.1%}) ===")
for i in fails[:5]:
    r = rec_by_idx.get(i)
    print(f"  idx {i} true={CLASS_NAMES[int(y_true[i])]} ens={CLASS_NAMES[int(ens_pred[i])]} "
          f"argus={CLASS_NAMES[int(argus_pred[i])]} JS={r['js']:.3f} fired={r['fired']} "
          f"conf={argus_confidences[i]:.3f}")


In [ ]:
# ============================================================================
# Cell 7b — Visualise the per-configuration metrics.
# ============================================================================
plot_df = metrics_table[["Balanced Accuracy", "Macro AUC", "ECE"]].astype(float)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, metric, better in zip(axes, plot_df.columns, ["higher", "higher", "lower"]):
    vals = plot_df[metric].values
    colors = sns.color_palette("viridis", len(vals))
    ax.bar(range(len(vals)), vals, color=colors)
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels(plot_df.index, rotation=30, ha="right", fontsize=9)
    ax.set_title(f"{metric}  ({better} is better)")
    for i, v in enumerate(vals):
        ax.text(i, v, f"{v:.3f}", ha="center", va="bottom", fontsize=8)
fig.suptitle("Argus Vision — metrics by configuration (full test split)", y=1.03)
fig.tight_layout()
fig.savefig(FIG_DIR / "metrics_by_configuration.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 6 — Metrics restricted to the hard subset $D_{hard}$

We load `hard_subset.csv` (NB03 output) via `find_file`. Because its `image_path` column was
written in a previous environment, we match its rows to our evaluated images by **filename stem**.
The same five configurations are then re-scored on just those hard cases.

In [ ]:
# ============================================================================
# Cell 8 — D_hard metrics table.
# ============================================================================
hard_csv = find_file("hard_subset")
eval_index_by_stem = {Path(p).stem: i for i, p in enumerate(image_paths)}

hard_indices: List[int] = []
if hard_csv is not None:
    hard_df = pd.read_csv(hard_csv)
    path_col = "image_path" if "image_path" in hard_df.columns else hard_df.columns[0]
    for raw in hard_df[path_col].astype(str):
        stem = Path(raw).stem
        if stem in eval_index_by_stem:
            hard_indices.append(eval_index_by_stem[stem])
    hard_indices = sorted(set(hard_indices))
    print(f"hard_subset.csv has {len(hard_df)} rows; "
          f"{len(hard_indices)} of them are in the current eval split.")
else:
    print("hard_subset.csv not found; falling back to images where the live debate trigger fired.")

# Fallback: use the images where our own trigger fired.
if len(hard_indices) == 0:
    hard_indices = [i for i, r in enumerate(debate_records) if r["fired"]]
    print(f"Using {len(hard_indices)} trigger-fired images as D_hard.")

if len(hard_indices) >= 2 and len(np.unique(y_true[hard_indices])) >= 2:
    idx = np.asarray(hard_indices)
    yh = y_true[idx]
    hard_rows = []
    for name in CONFIG_NAMES:
        m = metrics_for(proba[name][idx], yh)
        note = "SKIPPED" if (name == "Deep Ensemble" and not DEEP_ENSEMBLE_AVAILABLE) else ""
        hard_rows.append({"Configuration": name, **m, "Note": note})
    hard_metrics_table = pd.DataFrame(hard_rows).set_index("Configuration")
    hard_metrics_table.to_csv(WORK_DIR / "metrics_hard_subset.csv")
    disp = hard_metrics_table.copy()
    for c in ["Balanced Accuracy", "Macro AUC", "ECE"]:
        disp[c] = disp[c].map(lambda v: f"{v:.4f}")
    print(f"\nD_hard metrics over {len(idx)} images "
          f"(saved to /kaggle/working/metrics_hard_subset.csv):")
    display(disp)
else:
    print("Not enough hard-subset images (need >=2 with >=2 classes) to compute a D_hard table.")
    hard_metrics_table = None

## Step 7 - Ablation of the 23-dim feature groups

On the evaluated images, ablate feature groups by zeroing the raw feature before standardizing and re-running the same trained consensus MLP:

- **Full (23-dim)** - the complete vector.
- **No attention features** - dims 20-22 (attn IoU/entropies) set to 0.
- **Probabilities only** - dims 16-22 (disagreement + attention) set to 0, keeping pA, pB.

A drop when removing a group indicates that group carries signal beyond the raw agent probabilities.

In [ ]:
recs = [r for r in debate_records if "feat" in r and r.get("consensus_prob") is not None]
print(f"Ablation runs over {len(recs)} images with computed 23-dim features.")

if len(recs) >= 2:
    stem_to_label = {Path(p).stem: int(y_true[i]) for i, p in enumerate(image_paths)}
    feats_full = np.stack([r["feat"] for r in recs]).astype(np.float32)
    ab_labels = np.asarray([stem_to_label[r["image_id"]] for r in recs], dtype=np.int64)

    feats_noattn = feats_full.copy()
    feats_noattn[:, 20:23] = 0.0
    feats_probsonly = feats_full.copy()
    feats_probsonly[:, 16:23] = 0.0  # zero disagreement + attention, keep pA, pB

    def consensus_probs(raw_feats):
        Z = np.asarray(scaler.transform(raw_feats), dtype=np.float32)
        consensus.eval()
        with torch.no_grad():
            ft = torch.from_numpy(Z).to(DEVICE)
            return F.softmax(consensus(ft), dim=1).detach().cpu().numpy().astype(np.float64)

    variants = {
        "Full (23-dim)": feats_full,
        "No attention features (dims 20-22 = 0)": feats_noattn,
        "Probabilities only (dims 16-22 = 0)": feats_probsonly,
    }
    ab_rows = []
    for vname, fmat in variants.items():
        p = consensus_probs(fmat)
        m = metrics_for(p, ab_labels)
        ab_rows.append({"Ablation variant": vname, **m})
    ablation_table = pd.DataFrame(ab_rows).set_index("Ablation variant")
    ablation_table.to_csv(WORK_DIR / "ablation_table.csv")
    disp = ablation_table.copy()
    for c in ["Balanced Accuracy", "Macro AUC", "ECE"]:
        disp[c] = disp[c].map(lambda v: f"{v:.4f}")
    print("Ablation (saved to /kaggle/working/ablation_table.csv):")
    display(disp)
else:
    print("Too few records to run the ablation.")
    ablation_table = None


## Step 8 - Six case studies

Each case renders the original image, Agent A's Grad-CAM++, Agent B's attention rollout, the disagreement map M_delta, the confidence before (ensemble) vs after (Argus consensus), and a numeric summary of the 23-dim feature drivers (top class per agent, JS divergence, entropies, attention IoU). No debate text exists any more. Selected: agree-correct, debate-helps, debate-fails, and a melanoma case.

In [ ]:
# ============================================================================
# Cell 10 — Select the six case-study images.
# ============================================================================
stem_to_eval_idx = {Path(p).stem: i for i, p in enumerate(image_paths)}


def top_pred(prob_row: np.ndarray):
    return int(prob_row.argmax()), float(prob_row.max())


case_pool = []
for r in debate_records:
    i = stem_to_eval_idx[r["image_id"]]
    gt = int(y_true[i])
    a_pred = int(r["pa"].argmax()); b_pred = int(r["pb"].argmax())
    ens_pred = int(proba["Standard Ensemble"][i].argmax())
    argus_pred = int(proba["Argus (full)"][i].argmax())
    case_pool.append({
        "rec": r, "eval_idx": i, "gt": gt,
        "a_pred": a_pred, "b_pred": b_pred,
        "ens_pred": ens_pred, "argus_pred": argus_pred,
        "agree": a_pred == b_pred, "fired": r["fired"],
    })

selected = []
used = set()


def pick(predicate, count, label):
    chosen = []
    for c in case_pool:
        if len(chosen) >= count:
            break
        if c["eval_idx"] in used:
            continue
        if predicate(c):
            c2 = dict(c); c2["case_kind"] = label
            chosen.append(c2); used.add(c["eval_idx"])
    return chosen


# 1-2 agree-correct
selected += pick(lambda c: c["agree"] and c["a_pred"] == c["gt"], 2, "Agree-correct")
# 3-4 debate-helps: ensemble wrong, Argus right, and trigger fired
selected += pick(lambda c: c["fired"] and c["ens_pred"] != c["gt"] and c["argus_pred"] == c["gt"],
                 2, "Debate-helps")
# 5 debate-fails: trigger fired, Argus still wrong
selected += pick(lambda c: c["fired"] and c["argus_pred"] != c["gt"], 1, "Debate-fails")
# 6 melanoma (GT == MEL)
selected += pick(lambda c: c["gt"] == CLASS_NAMES.index("MEL"), 1, "Melanoma")

# Backfill to ensure we always show six cases even on degenerate splits.
fill_priority = sorted(case_pool, key=lambda c: (not c["fired"], not c["agree"]))
fi = 0
while len(selected) < 6 and fi < len(fill_priority):
    c = fill_priority[fi]; fi += 1
    if c["eval_idx"] in used:
        continue
    c2 = dict(c); c2["case_kind"] = "Additional"
    selected.append(c2); used.add(c["eval_idx"])

print(f"Selected {len(selected)} case studies:")
for c in selected:
    print(f"  [{c['case_kind']:13s}] {c['rec']['image_id']:>16s}  GT={CLASS_NAMES[c['gt']]:4s} "
          f"A={CLASS_NAMES[c['a_pred']]:4s} B={CLASS_NAMES[c['b_pred']]:4s} "
          f"Ens={CLASS_NAMES[c['ens_pred']]:4s} Argus={CLASS_NAMES[c['argus_pred']]:4s} "
          f"fired={c['fired']}")

In [ ]:
import textwrap


def denorm_for_show(image_path):
    with Image.open(image_path) as h:
        img = h.convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE))
    return np.asarray(img).astype(np.float32) / 255.0


def overlay(base_rgb, heat):
    hm = (np.clip(heat, 0, 1) * 255).astype(np.uint8)
    cmap = cv2.applyColorMap(hm, cv2.COLORMAP_JET)[:, :, ::-1].astype(np.float32) / 255.0
    return np.clip(0.55 * base_rgb + 0.45 * cmap, 0, 1)


for c in selected:
    r = c["rec"]; i = c["eval_idx"]
    base = denorm_for_show(r["image_path"])
    tensor = load_tensor(r["image_path"])
    hm_a = gradcam_plusplus(agent_a, tensor, int(r["pa"].argmax()))
    hm_b = attention_rollout(agent_b, tensor)
    m_delta = disagreement_map(hm_a, hm_b)

    fig = plt.figure(figsize=(16, 6.4))
    gs = fig.add_gridspec(2, 4, height_ratios=[1.0, 0.9])

    ax0 = fig.add_subplot(gs[0, 0]); ax0.imshow(base); ax0.axis("off")
    ax0.set_title(f"Original\nGT = {CLASS_NAMES[c['gt']]} ({FULL_NAMES[CLASS_NAMES[c['gt']]]})", fontsize=9)
    ax1 = fig.add_subplot(gs[0, 1]); ax1.imshow(overlay(base, hm_a)); ax1.axis("off")
    ax1.set_title(f"Agent A Grad-CAM++\npred {CLASS_NAMES[c['a_pred']]} ({r['pa'].max():.2f})", fontsize=9)
    ax2 = fig.add_subplot(gs[0, 2]); ax2.imshow(overlay(base, hm_b)); ax2.axis("off")
    ax2.set_title(f"Agent B attention\npred {CLASS_NAMES[c['b_pred']]} ({r['pb'].max():.2f})", fontsize=9)
    ax3 = fig.add_subplot(gs[0, 3]); im = ax3.imshow(m_delta, cmap="magma"); ax3.axis("off")
    ax3.set_title(r"Disagreement map $M_\Delta$", fontsize=9)
    fig.colorbar(im, ax=ax3, fraction=0.046, pad=0.04)

    ens_p = proba["Standard Ensemble"][i]; argus_p = proba["Argus (full)"][i]
    ens_top, ens_conf = top_pred(ens_p); arg_top, arg_conf = top_pred(argus_p)

    axc = fig.add_subplot(gs[1, 0])
    axc.bar([0, 1], [ens_conf, arg_conf], color=["#6B7280", "#2563EB"])
    axc.set_xticks([0, 1]); axc.set_xticklabels(["before\n(ensemble)", "after\n(Argus)"], fontsize=8)
    axc.set_ylim(0, 1); axc.set_ylabel("top-class conf", fontsize=8)
    axc.set_title(f"before={CLASS_NAMES[ens_top]} -> after={CLASS_NAMES[arg_top]}", fontsize=8)
    for xi, v in zip([0, 1], [ens_conf, arg_conf]):
        axc.text(xi, v, f"{v:.2f}", ha="center", va="bottom", fontsize=8)

    # Numeric 23-dim feature summary (replaces the removed debate text).
    summary = (
        f"23-dim feature drivers\n"
        f"-----------------------\n"
        f"Agent A top : {CLASS_NAMES[int(r['pa'].argmax())]} ({r['pa'].max():.3f})\n"
        f"Agent B top : {CLASS_NAMES[int(r['pb'].argmax())]} ({r['pb'].max():.3f})\n"
        f"JS divergence : {r['js']:.4f}\n"
        f"entropy A / B : {r['ent_a']:.2f} / {r['ent_b']:.2f} bits\n"
        f"attn IoU      : {r.get('attn_iou', 0.0):.3f}\n"
        f"trigger fired : {r['fired']}\n"
        f"Argus pred    : {CLASS_NAMES[c['argus_pred']]} ({arg_conf:.3f})"
    )
    axt = fig.add_subplot(gs[1, 1:]); axt.axis("off")
    axt.text(0.0, 1.0, summary, va="top", ha="left", fontsize=9, family="monospace")

    correct = "OK" if c["argus_pred"] == c["gt"] else "X"
    fig.suptitle(f"[{c['case_kind']}]  {r['image_id']}  |  JS={r['js']:.3f}  "
                 f"fired={r['fired']}  |  Argus={CLASS_NAMES[c['argus_pred']]} {correct}",
                 fontsize=11, y=1.02)
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"case_{c['case_kind'].lower().replace(' ', '_')}_{r['image_id']}.png",
                dpi=140, bbox_inches="tight")
    plt.show()

print("Case studies rendered and saved under /kaggle/working/figures/.")


## Step 9 — Confusion matrix for the full Argus system

A per-class confusion matrix (row-normalised recall) for the Argus configuration, to complement
the aggregate metrics above.

In [ ]:
# ============================================================================
# Cell 11 — Confusion matrix for the Argus (full) configuration.
# ============================================================================
from sklearn.metrics import confusion_matrix

argus_preds = proba["Argus (full)"].argmax(axis=1)
cm = confusion_matrix(y_true, argus_preds, labels=list(range(NUM_CLASSES)))
cm_norm = cm.astype(np.float64) / cm.sum(axis=1, keepdims=True).clip(min=1)

fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax,
            cbar_kws={"label": "recall (row-normalised)"})
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Argus (full) — confusion matrix on the test split")
fig.tight_layout()
fig.savefig(FIG_DIR / "argus_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## How to read this report & where outputs are saved

- **Step 5 (full-split table)** is the headline: Balanced Accuracy / Macro AUC (higher better), ECE (lower better) across the five configurations. Compare **Argus (full)** against the **Standard Ensemble**; Step 6's bootstrap CIs say whether any gap is statistically meaningful.
- **Phase 4** reports the clinically-motivated malignant-vs-benign recall/precision and the MEL->NV rate (the most dangerous confusion).
- **Phase 5** is the abstention trade-off: coverage vs selective balanced accuracy, with a recommended threshold.
- **Phase 6** gives 95% bootstrap CIs for Argus and the ensemble.
- **Phase 7** is the error analysis on the dangerous confusions and the debate-fail cases.
- **Step 7 (ablation)** attributes any Argus gain to the disagreement / attention feature groups over the raw agent probabilities.
- **Step 8 (case studies)** is the qualitative evidence; M_delta shows where the agents looked differently.

**Outputs** (under `/kaggle/working/`): `metrics_full_split.csv`, `metrics_hard_subset.csv`, `ablation_table.csv`, `malignant_vs_benign.csv`, `figures/abstention_curve.png`, `figures/argus_confusion_matrix.png`, `figures/case_*.png`.